# Mapa de rejeição

Agregue achados de lotes sintéticos por prestador e mês. Identificadores CPF são pseudonimizados com um segredo local.

In [1]:
from copy import deepcopy
from datetime import date
from pathlib import Path
from tempfile import TemporaryDirectory

from voa_tiss import examples, parse_lote, rejection_map
from voa_tiss._mapa_html import render_rejection_map_html

base = parse_lote(examples.load("lote-valido"))
for guia in base["guias"]:
    guia["procedimentos_executados"][0]["data_execucao"] = "2026-10-21"

lotes = [deepcopy(base) for _ in range(3)]
lotes[0]["prestador"] = {"codigo_prestador_na_operadora": "CLINICA-A"}
lotes[1]["prestador"] = {"cnpj": "11222333000181"}
lotes[1]["guias"][0]["procedimentos_executados"][0]["valor_total"] = 0
lotes[2]["prestador"] = {"cpf": "123.456.789-09"}

mapa = rejection_map(
    lotes,
    segredo_pseudonimo="segredo-sintetico-do-exemplo",
    at=date(2026, 10, 21),
)
linhas_resumo = [
    (linha["prestador"], linha["codigo"], linha["guias"], linha["valor_afetado"])
    for linha in mapa["linhas"]
]
linhas_resumo

[('cnpj:11222333000181', 'audit.valor_nao_positivo', 1, '0.00'),
 ('cnpj:11222333000181', 'item.valor_divergente', 1, '0.00'),
 ('cnpj:11222333000181', 'item.valor_zerado', 1, '100.00'),
 ('cnpj:11222333000181', 'lote.valor_total_geral', 1, '100.00'),
 ('cpf:hmac:f5c1b949ef3899e166f07590337717246bb7a89aaf6296b5c1944c2861ed99fe',
  'xsd.padrao',
  2,
  '200.00')]

A página HTML é estática e não depende de serviços externos. A CLI também lê XML e JSON recursivamente: `python -m voa_tiss mapa ./lotes --json mapa.json --html mapa.html`.

In [2]:
with TemporaryDirectory() as pasta_temporaria:
    pagina = Path(pasta_temporaria) / "mapa.html"
    pagina.write_text(render_rejection_map_html(mapa), encoding="utf-8")
    html_gerado = pagina.is_file()
html_gerado

True